**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Part 4 | Session 23: 그래프 RAG - 지식 그래프 기반 검색 증강

## 학습 목표

1️⃣ 벡터 RAG의 한계를 이해하고 그래프 RAG의 필요성을 파악한다  
2️⃣ 지식 그래프의 기본 개념 (노드, 엣지, 트리플)을 학습한다  
3️⃣ Graph RAG 파이프라인의 전체 흐름을 이해한다  
4️⃣ Neo4j를 활용한 그래프 RAG를 직접 구축한다  
5️⃣ LangChain GraphRAG 연동을 실습한다  

---

### 실습 환경
- **GPU**: 선택사항
- **필수 패키지**: langchain, neo4j, networkx, openai
- **외부 서비스**: Neo4j (Docker 또는 Neo4j Aura 무료 인스턴스)

In [ ]:
# 패키지 설치
!pip install -q langchain langchain-community langchain-openai neo4j networkx matplotlib openai tiktoken

In [ ]:
# 패키지 임포트 및 환경 설정
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple

# API 키 설정 (환경변수 또는 직접 입력)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["NEO4J_URI"] = "bolt://localhost:7687"
# os.environ["NEO4J_USERNAME"] = "neo4j"
# os.environ["NEO4J_PASSWORD"] = "password"

print("패키지 로드 완료")

---

## 1️⃣ 벡터 RAG의 한계

벡터 RAG는 문서를 벡터로 변환하여 유사도 기반 검색을 수행합니다.  
하지만 다음과 같은 구조적 한계가 존재합니다.

### 관계 정보 손실

벡터 임베딩은 텍스트의 **의미적 유사성**은 포착하지만, 엔티티 간 **명시적 관계**는 손실됩니다.

```
예시: "삼성전자의 CEO는 이재용이다" + "이재용은 서울대를 졸업했다"

벡터 RAG: 두 문장을 독립적인 청크로 저장 → 관계 연결 불가
그래프 RAG: 삼성전자 --[CEO]--> 이재용 --[졸업]--> 서울대 → 관계 추론 가능
```

### 멀티홉 추론의 어려움

| 질문 유형 | 벡터 RAG | 그래프 RAG |
|-----------|----------|------------|
| 단순 사실 질문 | ✅ 우수 | ✅ 우수 |
| 관계 기반 질문 | ❌ 한계 | ✅ 우수 |
| 멀티홉 추론 | ❌ 매우 어려움 | ✅ 자연스러움 |
| 글로벌 요약 | ❌ 한계 | ✅ 커뮤니티 요약 |

In [ ]:
# 벡터 RAG의 한계를 보여주는 간단한 예시

documents = [
    "삼성전자의 CEO는 이재용이다.",
    "이재용은 서울대학교 동양사학과를 졸업했다.",
    "서울대학교는 대한민국 서울시 관악구에 위치해 있다.",
    "삼성전자는 반도체와 스마트폰을 제조하는 기업이다.",
    "SK하이닉스의 CEO는 곽노정이다.",
]

# 멀티홉 질문: "삼성전자 CEO가 졸업한 대학은 어디에 있는가?"
# 이 질문에 답하려면 3개의 문서를 연결해야 합니다:
# 삼성전자 → CEO → 이재용 → 졸업 → 서울대 → 위치 → 관악구

print("멀티홉 질문: 삼성전자 CEO가 졸업한 대학은 어디에 있는가?")
print()
print("필요한 추론 경로:")
print("  삼성전자 --[CEO]--> 이재용")
print("  이재용 --[졸업]--> 서울대학교")
print("  서울대학교 --[위치]--> 서울시 관악구")
print()
print("벡터 RAG: 각 문서를 독립적으로 검색하여 연결이 어려움")
print("그래프 RAG: 관계 그래프를 따라 자연스럽게 추론 가능")

---

## 2️⃣ 지식 그래프 개요

### 기본 구성 요소

지식 그래프는 세 가지 핵심 요소로 구성됩니다:

- **노드 (Node)**: 엔티티를 나타냄 (사람, 조직, 장소 등)
- **엣지 (Edge)**: 노드 간의 관계를 나타냄 (소속, 위치, 생산 등)
- **트리플 (Triple)**: Subject - Predicate - Object 형태의 기본 단위

```
트리플 예시:
  (삼성전자, CEO, 이재용)
  (이재용, 졸업, 서울대학교)
  (서울대학교, 위치, 서울시 관악구)
```

### 지식 그래프 vs 일반 그래프

| 특성 | 일반 그래프 | 지식 그래프 |
|------|------------|------------|
| 노드 | 단순 식별자 | 타입이 있는 엔티티 |
| 엣지 | 단순 연결 | 의미 있는 관계 (레이블) |
| 속성 | 제한적 | 풍부한 프로퍼티 |
| 스키마 | 없음 | 온톨로지 기반 |

In [ ]:
# NetworkX로 간단한 지식 그래프 시각화

G = nx.DiGraph()

# 노드 추가 (엔티티)
entities = {
    "삼성전자": "기업",
    "이재용": "인물",
    "서울대학교": "대학",
    "관악구": "지역",
    "반도체": "제품",
    "SK하이닉스": "기업",
    "곽노정": "인물",
}

for entity, etype in entities.items():
    G.add_node(entity, type=etype)

# 엣지 추가 (관계 = 트리플)
triples = [
    ("삼성전자", "이재용", "CEO"),
    ("이재용", "서울대학교", "졸업"),
    ("서울대학교", "관악구", "위치"),
    ("삼성전자", "반도체", "생산"),
    ("SK하이닉스", "곽노정", "CEO"),
    ("SK하이닉스", "반도체", "생산"),
]

for subj, obj, pred in triples:
    G.add_edge(subj, obj, relation=pred)

# 시각화
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=2, seed=42)

# 노드 타입별 색상
color_map = {"기업": "#FF6B6B", "인물": "#4ECDC4", "대학": "#45B7D1", "지역": "#96CEB4", "제품": "#FFEAA7"}
node_colors = [color_map.get(entities.get(n, ""), "#DDA0DD") for n in G.nodes()]

nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=2000,
        font_size=10, font_family="NanumGothic", arrows=True, arrowsize=20,
        edge_color="gray", width=2)

# 엣지 레이블
edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9,
                              font_family="NanumGothic")

plt.title("지식 그래프 예시", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n노드 수: {G.number_of_nodes()}")
print(f"엣지 수: {G.number_of_edges()}")
print(f"트리플 목록:")
for subj, obj, pred in triples:
    print(f"  ({subj}, {pred}, {obj})")

---

## 3️⃣ Graph RAG 파이프라인

Graph RAG의 전체 파이프라인은 다음과 같습니다:

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────┐    ┌──────────────┐    ┌──────────────┐
│   문서 입력   │ →  │ 엔티티/관계 추출  │ →  │ 그래프 구축  │ →  │  그래프 검색   │ →  │  LLM 생성    │
└─────────────┘    └──────────────────┘    └─────────────┘    └──────────────┘    └──────────────┘
```

### 각 단계 상세

1. **문서 입력**: 원본 텍스트 문서를 청크 단위로 분할
2. **엔티티/관계 추출**: LLM을 사용하여 텍스트에서 엔티티와 관계를 추출
3. **그래프 구축**: 추출된 트리플을 그래프 데이터베이스에 저장
4. **그래프 검색**: 사용자 질문을 기반으로 관련 서브그래프 검색
5. **LLM 생성**: 검색된 그래프 컨텍스트를 기반으로 최종 답변 생성

In [ ]:
# Step 1 & 2: LLM을 활용한 엔티티/관계 추출

from openai import OpenAI

EXTRACTION_PROMPT = """다음 텍스트에서 엔티티(개체)와 관계를 추출하세요.
결과를 JSON 형식으로 반환하세요.

형식:
{{
  "entities": [
    {{"name": "엔티티명", "type": "유형(인물/기업/장소/제품 등)"}}
  ],
  "relations": [
    {{"subject": "주어 엔티티", "predicate": "관계", "object": "목적어 엔티티"}}
  ]
}}

텍스트: {text}
"""


def extract_entities_and_relations(text: str) -> Dict:
    """LLM을 사용하여 텍스트에서 엔티티와 관계를 추출합니다."""
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 지식 그래프 전문가입니다. 텍스트에서 엔티티와 관계를 정확하게 추출합니다."},
            {"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


# 예시 텍스트에서 추출
sample_text = """삼성전자는 1969년 설립된 대한민국의 대표적인 전자기업이다.
이재용 회장이 경영을 이끌고 있으며, 본사는 경기도 수원시에 위치해 있다.
삼성전자는 갤럭시 스마트폰과 DRAM 반도체를 주요 제품으로 생산한다.
2024년 매출은 약 260조원을 기록했다."""

print("엔티티/관계 추출 중...")
# result = extract_entities_and_relations(sample_text)  # API 키 필요

# 예시 결과 (API 호출 없이)
result = {
    "entities": [
        {"name": "삼성전자", "type": "기업"},
        {"name": "이재용", "type": "인물"},
        {"name": "수원시", "type": "장소"},
        {"name": "갤럭시", "type": "제품"},
        {"name": "DRAM", "type": "제품"},
    ],
    "relations": [
        {"subject": "삼성전자", "predicate": "설립연도", "object": "1969년"},
        {"subject": "이재용", "predicate": "경영", "object": "삼성전자"},
        {"subject": "삼성전자", "predicate": "본사위치", "object": "수원시"},
        {"subject": "삼성전자", "predicate": "생산", "object": "갤럭시"},
        {"subject": "삼성전자", "predicate": "생산", "object": "DRAM"},
    ]
}

print("\n추출된 엔티티:")
for e in result["entities"]:
    print(f"  - {e['name']} ({e['type']})")

print("\n추출된 관계 (트리플):")
for r in result["relations"]:
    print(f"  ({r['subject']}) --[{r['predicate']}]--> ({r['object']})")

In [ ]:
# Step 3: 추출된 트리플로 NetworkX 그래프 구축

def build_knowledge_graph(extraction_result: Dict) -> nx.DiGraph:
    """추출 결과를 기반으로 지식 그래프를 구축합니다."""
    G = nx.DiGraph()

    # 엔티티(노드) 추가
    for entity in extraction_result["entities"]:
        G.add_node(entity["name"], type=entity["type"])

    # 관계(엣지) 추가
    for relation in extraction_result["relations"]:
        G.add_edge(
            relation["subject"],
            relation["object"],
            relation=relation["predicate"]
        )

    return G


kg = build_knowledge_graph(result)
print(f"지식 그래프 구축 완료!")
print(f"  노드 수: {kg.number_of_nodes()}")
print(f"  엣지 수: {kg.number_of_edges()}")
print(f"\n노드 목록: {list(kg.nodes())}")
print(f"엣지 목록:")
for u, v, d in kg.edges(data=True):
    print(f"  {u} --[{d['relation']}]--> {v}")

In [ ]:
# Step 4: 그래프 검색 - 질문에서 엔티티를 추출하고 서브그래프 탐색

def search_graph(G: nx.DiGraph, query_entities: List[str], max_hops: int = 2) -> List[str]:
    """질문 관련 엔티티에서 시작하여 서브그래프를 탐색합니다."""
    relevant_triples = []
    visited = set()

    def bfs_search(start_node, depth=0):
        if depth >= max_hops or start_node in visited:
            return
        visited.add(start_node)

        # 나가는 엣지 탐색
        for _, target, data in G.out_edges(start_node, data=True):
            triple = f"({start_node}) --[{data['relation']}]--> ({target})"
            relevant_triples.append(triple)
            bfs_search(target, depth + 1)

        # 들어오는 엣지 탐색
        for source, _, data in G.in_edges(start_node, data=True):
            triple = f"({source}) --[{data['relation']}]--> ({start_node})"
            relevant_triples.append(triple)
            bfs_search(source, depth + 1)

    for entity in query_entities:
        if entity in G.nodes():
            bfs_search(entity)

    return list(set(relevant_triples))


# 질문: "삼성전자의 본사는 어디에 있고, 어떤 제품을 만드나요?"
query_entities = ["삼성전자"]
context_triples = search_graph(kg, query_entities, max_hops=2)

print("질문: 삼성전자의 본사는 어디에 있고, 어떤 제품을 만드나요?")
print(f"\n검색된 그래프 컨텍스트 ({len(context_triples)}개 트리플):")
for t in context_triples:
    print(f"  {t}")

In [ ]:
# Step 5: 그래프 컨텍스트를 활용한 LLM 답변 생성

GRAPH_RAG_PROMPT = """다음 지식 그래프 정보를 참고하여 질문에 답변하세요.

지식 그래프 컨텍스트:
{context}

질문: {question}

지식 그래프의 정보를 기반으로 정확하게 답변하세요."""


def graph_rag_answer(question: str, context_triples: List[str]) -> str:
    """그래프 컨텍스트를 사용하여 LLM 답변을 생성합니다."""
    context = "\n".join(context_triples)
    prompt = GRAPH_RAG_PROMPT.format(context=context, question=question)

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


# 실행 예시 (API 키 필요)
question = "삼성전자의 본사는 어디에 있고, 어떤 제품을 만드나요?"
# answer = graph_rag_answer(question, context_triples)
# print(answer)

print("Graph RAG 파이프라인 구성 완료!")
print(f"\n질문: {question}")
print(f"컨텍스트 트리플 수: {len(context_triples)}")
print("\n(API 키 설정 후 graph_rag_answer() 함수를 호출하면 답변이 생성됩니다)")

---

## 4️⃣ Neo4j 기반 그래프 RAG 실습

### Neo4j 소개

Neo4j는 가장 널리 사용되는 그래프 데이터베이스로, **Cypher** 쿼리 언어를 사용합니다.

```
Neo4j 설치 옵션:
1. Docker: docker run -p 7474:7474 -p 7687:7687 neo4j
2. Neo4j Aura: https://neo4j.com/cloud/aura-free/ (무료 클라우드)
3. Neo4j Desktop: 로컬 설치
```

### Cypher 쿼리 기초

```cypher
-- 노드 생성
CREATE (n:Person {name: '이재용', role: 'CEO'})

-- 관계 생성
MATCH (a:Person {name: '이재용'}), (b:Company {name: '삼성전자'})
CREATE (a)-[:WORKS_AT]->(b)

-- 검색
MATCH (p:Person)-[:WORKS_AT]->(c:Company)
WHERE c.name = '삼성전자'
RETURN p.name

-- 멀티홉 쿼리
MATCH (p:Person)-[:WORKS_AT]->(c:Company)-[:PRODUCES]->(prod:Product)
RETURN p.name, c.name, prod.name
```

In [ ]:
# Neo4j 연결 및 지식 그래프 구축

from neo4j import GraphDatabase


class Neo4jKnowledgeGraph:
    """Neo4j 기반 지식 그래프 관리 클래스"""

    def __init__(self, uri: str, username: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))
        print(f"Neo4j 연결 성공: {uri}")

    def close(self):
        self.driver.close()

    def clear_graph(self):
        """그래프 초기화"""
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
        print("그래프 초기화 완료")

    def add_entity(self, name: str, entity_type: str, properties: Dict = None):
        """엔티티(노드) 추가"""
        props = properties or {}
        props["name"] = name
        prop_str = ", ".join([f"{k}: ${k}" for k in props.keys()])
        query = f"MERGE (n:{entity_type} {{{prop_str}}})"
        with self.driver.session() as session:
            session.run(query, **props)

    def add_relation(self, subject: str, predicate: str, obj: str):
        """관계(엣지) 추가"""
        query = """
        MATCH (a {name: $subject}), (b {name: $object})
        MERGE (a)-[r:RELATION {type: $predicate}]->(b)
        """
        with self.driver.session() as session:
            session.run(query, subject=subject, object=obj, predicate=predicate)

    def query_subgraph(self, entity_name: str, max_hops: int = 2) -> List[Dict]:
        """특정 엔티티의 서브그래프 검색"""
        query = f"""
        MATCH path = (start {{name: $name}})-[*1..{max_hops}]-(connected)
        RETURN path
        LIMIT 50
        """
        results = []
        with self.driver.session() as session:
            records = session.run(query, name=entity_name)
            for record in records:
                results.append(str(record))
        return results

    def cypher_query(self, query: str, params: Dict = None) -> List[Dict]:
        """커스텀 Cypher 쿼리 실행"""
        with self.driver.session() as session:
            records = session.run(query, **(params or {}))
            return [dict(record) for record in records]


# 사용 예시 (Neo4j 실행 중일 때)
print("Neo4j 지식 그래프 클래스 정의 완료")
print()
print("사용 예시:")
print('  kg = Neo4jKnowledgeGraph("bolt://localhost:7687", "neo4j", "password")')
print('  kg.add_entity("삼성전자", "Company")')
print('  kg.add_entity("이재용", "Person")')
print('  kg.add_relation("이재용", "CEO", "삼성전자")')
print('  results = kg.query_subgraph("삼성전자")')

In [ ]:
# Neo4j에 대량 데이터 로드 및 그래프 기반 검색

# 한국 IT 기업 지식 그래프 데이터
kg_data = {
    "entities": [
        {"name": "삼성전자", "type": "Company", "props": {"founded": 1969, "industry": "전자"}},
        {"name": "SK하이닉스", "type": "Company", "props": {"founded": 1983, "industry": "반도체"}},
        {"name": "LG전자", "type": "Company", "props": {"founded": 1958, "industry": "전자"}},
        {"name": "네이버", "type": "Company", "props": {"founded": 1999, "industry": "인터넷"}},
        {"name": "카카오", "type": "Company", "props": {"founded": 2010, "industry": "인터넷"}},
        {"name": "이재용", "type": "Person", "props": {"role": "회장"}},
        {"name": "최수연", "type": "Person", "props": {"role": "CEO"}},
        {"name": "갤럭시", "type": "Product", "props": {"category": "스마트폰"}},
        {"name": "HBM", "type": "Product", "props": {"category": "반도체"}},
        {"name": "하이퍼클로바X", "type": "Product", "props": {"category": "AI모델"}},
    ],
    "relations": [
        {"subject": "이재용", "predicate": "CEO_OF", "object": "삼성전자"},
        {"subject": "최수연", "predicate": "CEO_OF", "object": "네이버"},
        {"subject": "삼성전자", "predicate": "PRODUCES", "object": "갤럭시"},
        {"subject": "SK하이닉스", "predicate": "PRODUCES", "object": "HBM"},
        {"subject": "네이버", "predicate": "DEVELOPS", "object": "하이퍼클로바X"},
        {"subject": "삼성전자", "predicate": "COMPETES_WITH", "object": "SK하이닉스"},
        {"subject": "네이버", "predicate": "COMPETES_WITH", "object": "카카오"},
    ]
}


def load_to_neo4j(neo4j_kg, data: Dict):
    """데이터를 Neo4j에 로드합니다."""
    neo4j_kg.clear_graph()

    for entity in data["entities"]:
        neo4j_kg.add_entity(entity["name"], entity["type"], entity.get("props", {}))
        print(f"  노드 추가: {entity['name']} ({entity['type']})")

    for rel in data["relations"]:
        neo4j_kg.add_relation(rel["subject"], rel["predicate"], rel["object"])
        print(f"  관계 추가: {rel['subject']} --[{rel['predicate']}]--> {rel['object']}")

    print(f"\n총 {len(data['entities'])}개 노드, {len(data['relations'])}개 관계 로드 완료")


# Neo4j 실행 중일 때 아래 주석을 해제하세요:
# kg = Neo4jKnowledgeGraph("bolt://localhost:7687", "neo4j", "password")
# load_to_neo4j(kg, kg_data)

print("한국 IT 기업 지식 그래프 데이터 준비 완료")
print(f"  엔티티: {len(kg_data['entities'])}개")
print(f"  관계: {len(kg_data['relations'])}개")

---

## 5️⃣ LangChain GraphRAG 연동

LangChain은 Neo4j와의 통합을 위한 다양한 도구를 제공합니다.

### 주요 컴포넌트

- **Neo4jGraph**: Neo4j 데이터베이스 연결 및 스키마 관리
- **GraphCypherQAChain**: 자연어 질문을 Cypher 쿼리로 변환
- **LLMGraphTransformer**: 텍스트에서 자동으로 그래프 추출

In [ ]:
# LangChain Neo4j 통합

from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from langchain_openai import ChatOpenAI


def setup_langchain_graph_rag():
    """LangChain 기반 Graph RAG를 설정합니다."""

    # Neo4j 연결
    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password=os.environ.get("NEO4J_PASSWORD", "password")
    )

    # 그래프 스키마 확인
    print("그래프 스키마:")
    print(graph.schema)

    # LLM 설정
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # GraphCypherQAChain: 자연어 → Cypher → 결과 → 자연어 답변
    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,
        allow_dangerous_requests=True
    )

    return chain


# Neo4j 실행 중일 때 아래 주석을 해제하세요:
# chain = setup_langchain_graph_rag()
# result = chain.invoke({"query": "삼성전자의 CEO는 누구인가요?"})
# print(result)

print("LangChain GraphCypherQAChain 설정 코드 준비 완료")
print()
print("동작 흐름:")
print("  1. 사용자 질문 입력")
print("  2. LLM이 질문을 Cypher 쿼리로 변환")
print("  3. Neo4j에서 Cypher 쿼리 실행")
print("  4. 쿼리 결과를 LLM에 전달")
print("  5. LLM이 자연어 답변 생성")

In [ ]:
# LLMGraphTransformer: 텍스트에서 자동으로 그래프 추출

from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document


def auto_extract_graph(texts: List[str]):
    """LLMGraphTransformer를 사용하여 텍스트에서 자동으로 그래프를 추출합니다."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    transformer = LLMGraphTransformer(llm=llm)

    documents = [Document(page_content=text) for text in texts]
    graph_documents = transformer.convert_to_graph_documents(documents)

    print(f"\n추출된 그래프 문서: {len(graph_documents)}개")
    for i, doc in enumerate(graph_documents):
        print(f"\n--- 문서 {i+1} ---")
        print(f"  노드: {[str(n) for n in doc.nodes]}")
        print(f"  관계: {[str(r) for r in doc.relationships]}")

    return graph_documents


# 사용 예시
sample_texts = [
    "네이버의 CEO 최수연은 하이퍼클로바X 개발을 주도하고 있다.",
    "카카오는 카카오톡과 카카오맵 등 다양한 서비스를 제공하는 IT 기업이다.",
]

# API 키 설정 후 실행:
# graph_docs = auto_extract_graph(sample_texts)

print("LLMGraphTransformer 자동 그래프 추출 코드 준비 완료")
print("\n이 기능은 다음을 자동으로 수행합니다:")
print("  1. 텍스트에서 엔티티 식별")
print("  2. 엔티티 간 관계 추출")
print("  3. 구조화된 그래프 문서 생성")
print("  4. Neo4j에 직접 저장 가능")

---

## 정리

### 핵심 요약

| 항목 | 벡터 RAG | 그래프 RAG |
|------|----------|------------|
| 데이터 구조 | 벡터 임베딩 | 지식 그래프 (트리플) |
| 검색 방식 | 유사도 검색 | 그래프 탐색 (BFS/DFS) |
| 강점 | 의미적 유사성 | 관계 추론, 멀티홉 |
| 약점 | 관계 정보 손실 | 그래프 구축 비용 |
| 적합한 질문 | 단순 사실 질문 | 관계/추론 질문 |

### 다음 단계

- 온톨로지 RAG: 도메인 지식을 체계적으로 정의하여 더 강력한 추론을 수행
- 하이브리드 RAG: 벡터 검색 + 그래프 검색을 결합한 최적의 접근
- Microsoft GraphRAG: 커뮤니티 기반 글로벌/로컬 검색

In [ ]:
print("=" * 60)
print("그래프 RAG 실습 완료!")
print("=" * 60)
print()
print("학습 내용 정리:")
print("  1. 벡터 RAG의 한계: 관계 정보 손실, 멀티홉 추론 어려움")
print("  2. 지식 그래프: 노드, 엣지, 트리플 (SPO)")
print("  3. Graph RAG 파이프라인: 추출 → 구축 → 검색 → 생성")
print("  4. Neo4j + Cypher 쿼리로 그래프 RAG 구현")
print("  5. LangChain GraphCypherQAChain 및 LLMGraphTransformer 활용")